# Async: نمونه ناهمگام

داده‌های این نمونه ساختگی هستند؛ قیمت واقعی بازار نیستند.

## سلول ۱ — ورود کتابخانه‌ها

In [ ]:
from pathlib import Path
import tempfile
import asyncio
import pandas as pd
from pydara import tse
from pydara.tse.samples import sample_async_client

## سلول ۲ — تعریف توابع

In [ ]:
async def export_in_worker(frame, path):
    await asyncio.to_thread(tse.save, frame, path)
    restored = await asyncio.to_thread(tse.load, path)
    pd.testing.assert_frame_equal(frame, restored)
    return restored

## سلول ۳ — تنظیمات

In [ ]:
SYMBOL = 'فولاد'
OUTPUT = Path(tempfile.mkdtemp(prefix="pydara-part3-async-"))

## سلول ۴ — اجرا

In [ ]:
async with sample_async_client(storage=True, data_dir=OUTPUT) as client:
    sync_result = await client.sync_history(SYMBOL, timeout=30, on_storage_error="raise")
    df = await client.load_stored_history(SYMBOL)
    restored = await export_in_worker(df, OUTPUT / "history.csv")

## سلول ۵ — گزارش نتیجه

In [ ]:
display(pd.DataFrame([sync_result.summary]))
display(restored[["date", "closing_price"]].tail())